In [ ]:
# Install the required libraries
!pip install -qU pydantic langchain langchain-openai faiss-cpu langchain_community

**Importing Packages**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI
from typing import List, Dict, Any
import faiss
import pickle
import os
from getpass import getpass

In [ ]:
# Prompt for the OpenAI API Key securely
API_KEY = getpass("Enter your OpenAI API Key: ")

# Optional: required only for institutional/proxy endpoints
base_url = getpass(
    "Enter OPENAI_BASE_URL (press Enter if not required): "
).strip()

if base_url:
  client = OpenAI(api_key=API_KEY, base_url=base_url)
else:
  client = OpenAI(api_key=API_KEY)

EMBED_MODEL = "text-embedding-3-small"

**Managing the Directories Paths**

In [ ]:
# Set all path variables
PROJECT_PATH = Path("/content/capstone_project/")
PROCESSED_DIR = PROJECT_PATH / "data" / "processed"
PRODUCT_FAISS_INDEX_PATH = PROJECT_PATH / "outputs" / "vector_stores" / "product_faiss_index"

**Loading of the Processed Retail Dataset**

In [ ]:
# Reading/loading of the raw Data CSV file
processed_csv = PROCESSED_DIR / "retail_processed.csv"
retail_processed_df = pd.read_csv(processed_csv)

In [ ]:
retail_processed_df.info()

In [ ]:
# Converting each products from the catalog into a Document Sring
def product_to_text(row: pd.Series) -> str:
    return (
        f"Product Title: {row['product_title']}\n"
        f"Product Description: {row['product_description']}\n"
        f"Customer Review: {row['customer_review_text']}\n"
        f"Category: {row['category']}\n"
        f"Sub-Category: {row['sub_category']}\n"
        f"Brand: {row['brand']}\n"
        f"Base Price: ${row['base_price']}\n"
        f"Final Price: ${row['final_price']}\n"
        f"Discount Percent: ${row['discount_percent']}\n"
        f"Color: {row['color']}\n"
        f"Style Tags: {row['style_tags']}\n"
        f"Occasion Tags: {row['occasion_tags']}\n"
        f"Season: {row['season']}"
    )

In the processed CSV, data contains customer/product interaction rows. Those, not necessarily have one unique row per product.
While creating the Document object, we will not create the document per row from the CSV rather we will calculate unique product-ids and create document per unique product

In [ ]:
print("Total rows in CSV :", len(retail_processed_df))
print("Unique products out of it :", retail_processed_df["product_id"].nunique())

In [ ]:
# Create a unique product DataFrame
retail_processed_df_products = retail_processed_df.drop_duplicates(
    subset=["product_id"]
).copy()

In [ ]:
# Re-calculating the Product rows
print("Product Rows :", len(retail_processed_df_products))
print("Unique products out of it :", retail_processed_df_products["product_id"].nunique())

In [ ]:
retail_processed_df_products["doc"] = retail_processed_df_products.apply(product_to_text, axis=1)

In [ ]:
retail_processed_df_products.head()

In [ ]:
# Converting Text into the Langchain Document
documents = []
for _,row in retail_processed_df_products.iterrows():
  documents.append(
      Document(
        page_content=row['doc'],
         metadata={
        "product_id": row["product_id"],
        "product_title": row["product_title"],
        "category": row["category"],
        "sub_category": row["sub_category"],
        "brand": row["brand"],
        "base_price": row["base_price"],
        "final_price": row["final_price"],
        "discount_percent": row["discount_percent"],
        "avg_product_rating": row["avg_product_rating"],
        "stock_level": row["stock_level"],
        "is_returnable": row["is_returnable"],
        "image_path": row["image_path"]
    }
      )
  )

A pandas DF has been created and each product row is transformed into a Langchain Document

**Text RAG Retrieval (Core)**

In [ ]:
# Implementing Embeddings and crate the vector store
def embed_texts(texts: List[str], model: str = EMBED_MODEL) -> np.ndarray:
    # TODO: Implement embeddings call using client.embeddings.create
    resp = client.embeddings.create(model=model, input=texts)
    vecs = [d.embedding for d in resp.data]
    return np.array(vecs, dtype=np.float32)

In [ ]:
docs = retail_processed_df_products["doc"].tolist()
emb = embed_texts(docs, EMBED_MODEL)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)

# TODO: normalize and add to index
faiss.normalize_L2(emb)
index.add(emb)

print("Index ready:", index.ntotal, "vectors | dim:", dim)

In [ ]:
def retrieve_products(query: str, k: int = 5) -> List[Dict[str, Any]]:
  qv = embed_texts([query], EMBED_MODEL)
  faiss.normalize_L2(qv)

  scores, ids = index.search(qv, k)
  results = []
  for score, idx in zip(scores[0], ids[0]):
      row = retail_processed_df_products.iloc[int(idx)]
      results.append({
          "product_id": row["product_id"],
          "product_title": row["product_title"],
          "category": row["category"],
          "sub_category": row["sub_category"],
          "brand": row["brand"],
          "base_price": row["base_price"],
          "final_price": row["final_price"],
          "discount_percent": row["discount_percent"],
          "avg_product_rating": row["avg_product_rating"],
          "stock_level": row["stock_level"],
          "is_returnable": row["is_returnable"],
          "image_path": row["image_path"],
          "doc": row["doc"]
      })
  return results

In [ ]:
# Time to verify if the Retriever tool working as expected
retriver = retrieve_products("recommend good running shoes under 1000", k=3)
print(retriver)

**Saving the Product Index**

In [ ]:
file_path = os.path.join(PRODUCT_FAISS_INDEX_PATH, "index.faiss")
faiss.write_index(index, file_path)
print(f"Index safely saved to: {file_path}")

Because we are using the native faiss library to write the Index, the original texts are not saved in 'index.faiss', we should save our text/metadata arrays in the exact same directory using **pickle** in the form of proper LangChain Document objects

In [ ]:
# Transform raw text strings into LangChain Document objects before saving those in the pickle
# (If our 'docs' are already a list of Documents, this line will safely preserve them)
processed_docs = [
    doc if isinstance(doc, Document) else Document(page_content=str(doc))
    for doc in docs
]

# Save texts alongside the vectors
docs_path = os.path.join(PRODUCT_FAISS_INDEX_PATH, "docs.pkl")
with open(docs_path, "wb") as f:
    pickle.dump(processed_docs, f)

# **Conclusion And Outcome**

In this notebook, I have implemented the Product RAG component of the  Retail Assistant for catalog-grounded product recommendations.

**Activities Performed:**

* Prepared product catalog records as searchable product documents.
* Generated embeddings using the selected embedding model.
* Built a FAISS vector index using cosine-similarity-based retrieval.
* Implemented semantic retrieval for product-related user queries.
* Tested retrieval using representative recommendation and product-search queries.


**Output :**

Product FAISS index: outputs/vector_stores/product_faiss_index/